# LinkedIn Financial Planner Lead Scraper
## סורק לינקדין לאיתור לידים של מתכננים פיננסיים

**Flow:**
1. הגדרת פרטי כניסה ל-LinkedIn
2. הרצת הסריקה (לפי מילות חיפוש)
3. צפייה בתוצאות וסינון לידים איכותיים
4. ייצוא ל-CSV

---
> **Note:** Use this tool responsibly and in compliance with LinkedIn's Terms of Service.

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'playwright', 'pandas', 'ipywidgets'])
subprocess.check_call(['playwright', 'install', 'chromium', '--with-deps'])
print('Dependencies ready.')

In [ ]:
# ── Step 1: Configure credentials ────────────────────────────────────────
import os

# Option A: set here directly (not recommended for shared notebooks)
# os.environ['LINKEDIN_EMAIL']    = 'your@email.com'
# os.environ['LINKEDIN_PASSWORD'] = 'yourpassword'

# Option B: load from .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('Loaded .env')
except ImportError:
    pass

print('LINKEDIN_EMAIL set:', bool(os.getenv('LINKEDIN_EMAIL')))
print('LINKEDIN_PASSWORD set:', bool(os.getenv('LINKEDIN_PASSWORD')))
print()
print('Tip: on first run use --no-headless (visible browser) to handle CAPTCHA.')
print('After a successful login the session cookie is saved automatically.')

In [ ]:
# ── Step 2: Choose search keywords ───────────────────────────────────────
import linkedin_config as cfg

# Customise as needed – defaults come from linkedin_config.py
KEYWORDS = [
    'financial planner',
    'certified financial planner CFP',
    'financial advisor',
    'wealth manager',
    'מתכנן פיננסי',
    'יועץ פיננסי',
]

MAX_PER_KEYWORD = 15   # profiles to collect per keyword
MIN_SCORE       = 25   # only keep leads with score >= this
HEADLESS        = True # set False to see the browser window

print(f'Keywords: {KEYWORDS}')
print(f'Max per keyword: {MAX_PER_KEYWORD}')
print(f'Min lead score: {MIN_SCORE}')

In [ ]:
# ── Step 3: Run the scraper ───────────────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()   # allow asyncio.run() inside Jupyter

from linkedin_scraper import run_scraper
import asyncio

storage = asyncio.run(
    run_scraper(
        keywords=KEYWORDS,
        max_per_keyword=MAX_PER_KEYWORD,
        headless=HEADLESS,
        min_score=MIN_SCORE,
    )
)

summary = storage.summary()
print(f"\n✅ Total leads saved: {summary['total_leads']}")
print(f"   Average score:      {summary['avg_score']}")

In [ ]:
# ── Step 4: View results as a DataFrame ──────────────────────────────────
import pandas as pd
import sqlite3
import linkedin_config as cfg

con = sqlite3.connect(cfg.OUTPUT_DB)
df  = pd.read_sql('SELECT * FROM leads ORDER BY lead_score DESC', con)
con.close()

print(f'Total leads: {len(df)}')
df[['lead_score','name','title','location','connections','avg_comments','avg_likes','email','website']].head(20)

In [ ]:
# ── Step 5: Filter high-quality leads ────────────────────────────────────
hot_leads = df[
    (df['lead_score'] >= 40) &
    (df['avg_comments'] >= 5) &
    (df['connections'] >= 500)
].sort_values('lead_score', ascending=False)

print(f'High-quality leads (score>=40, avg_comments>=5, connections>=500): {len(hot_leads)}')
hot_leads[['lead_score','name','title','connections','avg_comments','profile_url']].head(20)

In [ ]:
# ── Step 6: Quick engagement distribution ────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

df['lead_score'].hist(ax=axes[0], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Lead Score Distribution')
axes[0].set_xlabel('Score')

df['avg_comments'].clip(upper=50).hist(ax=axes[1], bins=20, color='orange', edgecolor='white')
axes[1].set_title('Avg Comments per Post')
axes[1].set_xlabel('Comments')

df['connections'].clip(upper=10000).hist(ax=axes[2], bins=20, color='green', edgecolor='white')
axes[2].set_title('Connections / Followers')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# ── Step 7: Export to CSV ─────────────────────────────────────────────────
import linkedin_config as cfg

csv_path = storage.export_csv(min_score=MIN_SCORE)
print(f'Exported to: {csv_path}')

# Also save hot leads to a separate file
hot_csv = cfg.OUTPUT_DIR + '/hot_leads.csv'
hot_leads.to_csv(hot_csv, index=False)
print(f'Hot leads exported to: {hot_csv}')

In [ ]:
# ── Step 8: Preview top leads with post info ──────────────────────────────
cols = ['lead_score','name','title','connections','avg_comments',
        'avg_likes','top_post_comments','email','website','profile_url']
top10 = df[cols].head(10)

# Make profile_url clickable
def make_link(url):
    return f'<a href="{url}" target="_blank">{url[:40]}...</a>' if pd.notna(url) else ''

from IPython.display import HTML
styled = top10.copy()
styled['profile_url'] = styled['profile_url'].apply(make_link)
HTML(styled.to_html(escape=False, index=False))